In [1]:
# reliability.py
# Analytical reliability functions and simple system-level calculations for component-level analysis
# Dependencies: numpy, scipy

import numpy as np
from math import exp, gamma, prod, comb
from scipy import stats
from scipy.optimize import minimize

# -------------------------
# Component reliability models
# -------------------------

def R_exponential(lmbda, t):
    """Exponential survival: R(t) = exp(-lambda * t).
    lmbda: failure rate (1/time)
    t: time (scalar or array)
    """
    t = np.asarray(t)
    return np.exp(-lmbda * t)

def mttf_exponential(lmbda):
    """Mean time to failure for exponential (1/lambda)."""
    return 1.0 / lmbda

def hazard_exponential(lmbda, t=None):
    """Hazard for exponential: constant lambda."""
    return lmbda

def R_weibull(beta, eta, t):
    """Weibull survival: R(t) = exp(-(t/eta)**beta)
    beta: shape, eta: scale (same units as t)
    """
    t = np.asarray(t)
    return np.exp(- (t / eta) ** beta)

def hazard_weibull(beta, eta, t):
    """Weibull hazard: h(t) = (beta/eta) * (t/eta)^(beta-1)."""
    t = np.asarray(t)
    # handle t == 0 for beta < 1: hazard tends to infinity for beta < 1 at t->0, but return 0 at exact 0 for numeric stability
    with np.errstate(divide='ignore', invalid='ignore'):
        h = (beta / eta) * (t / eta) ** (beta - 1)
        h = np.where(t == 0, 0.0 if beta > 1 else h, h)
    return h

def mttf_weibull(beta, eta):
    """MTTF for Weibull: eta * Gamma(1 + 1/beta)."""
    return eta * gamma(1.0 + 1.0 / beta)

def R_lognormal(mu, sigma, t):
    """Lognormal survival: R(t) = 1 - Phi((ln t - mu)/sigma) for t>0."""
    t = np.asarray(t)
    z = (np.log(t) - mu) / sigma
    return 1.0 - stats.norm.cdf(z)

def hazard_lognormal(mu, sigma, t):
    """Approximate hazard for lognormal: f(t)/R(t)."""
    t = np.asarray(t)
    with np.errstate(divide='ignore', invalid='ignore'):
        pdf = (1/(t*sigma*np.sqrt(2*np.pi))) * np.exp(-0.5 * ((np.log(t)-mu)/sigma)**2)
        R = R_lognormal(mu, sigma, t)
        h = pdf / R
        h = np.where(t <= 0, 0.0, h)
    return h

# -------------------------
# Simple system reliability combinators
# -------------------------

def series_system_reliability(Rs):
    """Series system: product of component reliabilities Rs (iterable)."""
    Rs = np.asarray(Rs)
    return np.prod(Rs)

def parallel_system_reliability(Rs):
    """Parallel system: 1 - product(1 - R_i)."""
    Rs = np.asarray(Rs)
    return 1.0 - np.prod(1.0 - Rs)

def k_of_n_reliability(component_Rs, k):
    """General k-out-of-n reliability for components with possibly different R_i.
    Computes the probability that at least k components are functioning at mission time.
    This sums over combinations (works for n up to maybe ~20 depending on performance).
    component_Rs: iterable of reliability probabilities for each independent component
    """
    Rs = np.asarray(component_Rs)
    n = Rs.size
    if k <= 0:
        return 1.0
    if k > n:
        return 0.0
    # For different R_i we sum over all subsets where count(success) >= k:
    # P = sum_{subset S, |S|>=k} (prod_{i in S} R_i * prod_{j not in S} (1-R_j))
    total = 0.0
    # iterate masks 0..2^n-1
    for mask in range(1 << n):
        success_count = bin(mask).count("1")
        if success_count >= k:
            p = 1.0
            for i in range(n):
                if (mask >> i) & 1:
                    p *= Rs[i]
                else:
                    p *= (1.0 - Rs[i])
            total += p
    return total

# -------------------------
# Component importance measures
# -------------------------

def birnbaum_importance(system_func, Rs, i, *args, **kwargs):
    """Birnbaum importance for component i:
    I_B(i) = P(system works | comp i works) - P(system works | comp i fails)
    system_func: function that accepts component reliability array and returns system reliability (scalar)
    Rs: array of component reliabilities
    i: index of component (0-based)
    *args, **kwargs forwarded to system_func
    """
    Rs = np.array(Rs, dtype=float)
    # Force component i to work (R=1) and to fail (R=0)
    Rs_force1 = Rs.copy()
    Rs_force1[i] = 1.0
    Rs_force0 = Rs.copy()
    Rs_force0[i] = 0.0
    p1 = system_func(Rs_force1, *args, **kwargs)
    p0 = system_func(Rs_force0, *args, **kwargs)
    return p1 - p0

# Fast closed-form Birnbaum for series/parallel:
def birnbaum_series(Rs, i):
    """For series system: Birnbaum_i = product_{j != i} R_j"""
    Rs = np.asarray(Rs)
    others = np.delete(Rs, i)
    return np.prod(others)

def birnbaum_parallel(Rs, i):
    """For parallel system: Birnbaum_i = product_{j != i} (1 - R_j)"""
    Rs = np.asarray(Rs)
    others = np.delete(Rs, i)
    return np.prod(1.0 - others)

# -------------------------
# Parameter estimation (Weibull MLE for uncensored failure times)
# -------------------------

def fit_weibull_mle(failure_times, initial=None):
    """Fit Weibull (beta, eta) by MLE for uncensored failure times.
    failure_times: 1D array-like of failure times > 0
    initial: optional tuple (beta0, eta0). If None, an initial guess is computed.
    Returns dict: {'beta':..., 'eta':..., 'success':True/False}
    """
    t = np.asarray(failure_times, dtype=float)
    if np.any(t <= 0):
        raise ValueError("Failure times must be > 0 for log-likelihood")

    # compute default initial guess inside function to avoid NameError at definition time
    if initial is None:
        # reasonable defaults: beta ~ 1.5, eta ~ mean(t)
        initial = (1.5, float(np.mean(t)))

    def neg_loglik(params):
        beta, eta = params
        if beta <= 0 or eta <= 0:
            return 1e20
        # log-likelihood for uncensored Weibull
        ll = np.sum(np.log(beta) - beta * np.log(eta) + (beta - 1) * np.log(t) - (t / eta) ** beta)
        return -ll

    res = minimize(neg_loglik, x0=np.array(initial), bounds=[(1e-6, None), (1e-6, None)], method='L-BFGS-B')
    if res.success:
        beta_hat, eta_hat = res.x
        return {'beta': float(beta_hat), 'eta': float(eta_hat), 'success': True}
    else:
        return {'beta': None, 'eta': None, 'success': False, 'message': res.message}

# -------------------------
# Utility: system reliability from boolean structure function (truth-table approach)
# -------------------------
def system_reliability_from_structure_fn(Rs, structure_fn):
    """Compute system reliability when structure_fn(x) returns 1 if system works given binary vector x of component states.
    Rs: iterable of component reliabilities (probabilities component is working)
    structure_fn: callable taking a binary list/array of component states (0/1) and returning 0/1 for system state.
    This enumerates all 2^n states; use n small (~<=20).
    """
    Rs = np.asarray(Rs, dtype=float)
    n = Rs.size
    total = 0.0
    for mask in range(1 << n):
        x = [(mask >> i) & 1 for i in range(n)]
        p = 1.0
        for i, xi in enumerate(x):
            p *= Rs[i] if xi else (1.0 - Rs[i])
        sys_ok = structure_fn(x)
        if sys_ok:
            total += p
    return total

In [2]:
# Example usage of reliability functions

import numpy as np

# Exponential distribution example
lmbda = 0.01  # failure rate per unit time
t = 100  # time
print(f"Exponential reliability at t={t}: R(t) = {R_exponential(lmbda, t):.4f}")
print(f"Mean time to failure (MTTF): {mttf_exponential(lmbda):.2f}")
print(f"Hazard rate: {hazard_exponential(lmbda):.4f}")

# Weibull distribution example
beta = 2.0  # shape parameter
eta = 1000  # scale parameter
print(f"\nWeibull reliability at t={t}: R(t) = {R_weibull(beta, eta, t):.4f}")
print(f"MTTF: {mttf_weibull(beta, eta):.2f}")
print(f"Hazard at t={t}: {hazard_weibull(beta, eta, t):.6f}")

# System reliability examples
Rs = [0.9, 0.95, 0.85]  # component reliabilities
print(f"\nComponent reliabilities: {Rs}")
print(f"Series system reliability: {series_system_reliability(Rs):.4f}")
print(f"Parallel system reliability: {parallel_system_reliability(Rs):.4f}")
print(f"2-out-of-3 system reliability: {k_of_n_reliability(Rs, 2):.4f}")

# Birnbaum importance for series system
for i in range(len(Rs)):
    imp = birnbaum_series(Rs, i)
    print(f"Birnbaum importance of component {i+1} in series: {imp:.4f}")

# Weibull MLE fit example (simulated data)
np.random.seed(42)
failure_times = np.random.weibull(beta, 100) * eta  # simulate 100 failures
fit_result = fit_weibull_mle(failure_times)
if fit_result['success']:
    print(f"\nWeibull MLE fit: beta={fit_result['beta']:.2f}, eta={fit_result['eta']:.2f}")
else:
    print("\nWeibull MLE fit failed")

# Structure function example: bridge system (simplified)
def bridge_structure(x):
    # Bridge: works if (1 and 4) or (2 and 5) or (1 and 3 and 5) or (2 and 3 and 4)
    # Simplified: assume 5 components
    if len(x) != 5:
        return 0
    return int((x[0] and x[3]) or (x[1] and x[4]) or (x[0] and x[2] and x[4]) or (x[1] and x[2] and x[3]))

Rs_bridge = [0.9, 0.95, 0.85, 0.92, 0.88]
bridge_R = system_reliability_from_structure_fn(Rs_bridge, bridge_structure)
print(f"\nBridge system reliability: {bridge_R:.4f}")

Exponential reliability at t=100: R(t) = 0.3679
Mean time to failure (MTTF): 100.00
Hazard rate: 0.0100

Weibull reliability at t=100: R(t) = 0.9900
MTTF: 886.23
Hazard at t=100: 0.000200

Component reliabilities: [0.9, 0.95, 0.85]
Series system reliability: 0.7268
Parallel system reliability: 0.9992
2-out-of-3 system reliability: 0.9740
Birnbaum importance of component 1 in series: 0.8075
Birnbaum importance of component 2 in series: 0.7650
Birnbaum importance of component 3 in series: 0.8550

Weibull MLE fit: beta=1.93, eta=948.57

Bridge system reliability: 0.9834
